In [1]:
import sys
sys.path.append("..")

import os
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_sample_weight

from src.features import build_features

d = np.load("../data/processed/beats.npz")
CLASSES = ["N", "S", "V", "F"]


def load(split):
    X, y, rr, g = d[f"X_{split}"], d[f"y_{split}"], d[f"rr_{split}"], d[f"g_{split}"]
    m = y != 4                       # drop Q
    return build_features(X[m], rr[m], g[m]), y[m], g[m]


F_train, y_train, g_train = load("train")
F_test, y_test, g_test = load("test")
print(F_train.shape, F_test.shape)

gkf = GroupKFold(n_splits=5)


def make_rf(n_trees, leaf):
    return RandomForestClassifier(n_estimators=n_trees, min_samples_leaf=leaf,
                                  n_jobs=-1, random_state=42)


def oof_proba(n_trees, leaf):
    proba = np.zeros((len(y_train), 4))
    for tr, va in gkf.split(F_train, y_train, g_train):
        m = make_rf(n_trees, leaf)
        m.fit(F_train[tr], y_train[tr],
              sample_weight=compute_sample_weight("balanced", y_train[tr]))
        p = np.zeros((len(va), 4))
        p[:, m.classes_] = m.predict_proba(F_train[va])
        proba[va] = p
    return proba


def tune_sv(proba, y, grid=(0.5, 1, 2, 3, 4, 6, 8, 12)):
    """Pick decision weights for S and V (N=1, F=0) that maximize macro F1 over N,S,V."""
    best, best_w = -1, None
    for ws in grid:
        for wv in grid:
            w = np.array([1.0, ws, wv, 0.0])
            s = f1_score(y, (proba * w).argmax(1), labels=[0, 1, 2], average="macro")
            if s > best:
                best, best_w = s, w
    return best_w, best


CONFIGS = [(200, 2), (100, 5), (50, 5), (50, 20), (30, 20), (30, 50)]
rows = []
for n_trees, leaf in CONFIGS:
    p = oof_proba(n_trees, leaf)
    w, s = tune_sv(p, y_train)
    model = make_rf(n_trees, leaf).fit(
        F_train, y_train, sample_weight=compute_sample_weight("balanced", y_train))
    nodes = sum(t.tree_.node_count for t in model.estimators_)
    rows.append({"trees": n_trees, "leaf": leaf, "nodes": nodes, "cv": s, "w": w, "model": model})
    print(f"trees={n_trees:3d} leaf={leaf:2d} | nodes={nodes:8d} "
          f"| CV macro F1 (N,S,V)={s:.3f} | weights={w}")

# smallest model whose CV score is within 0.01 of the best one
best_cv = max(r["cv"] for r in rows)
chosen = min((r for r in rows if r["cv"] >= best_cv - 0.01), key=lambda r: r["nodes"])
print("\nchosen: trees =", chosen["trees"], "| leaf =", chosen["leaf"],
      "| nodes =", chosen["nodes"], "| weights =", chosen["w"])
print("approx. size as compact binary for the phone:",
      round(chosen["nodes"] * 22 / 1e6, 2), "MB")

# one final look at the test patients
model, w = chosen["model"], chosen["w"]
proba = np.zeros((len(y_test), 4))
proba[:, model.classes_] = model.predict_proba(F_test)
pred = (proba * w).argmax(1)
print(classification_report(y_test, pred, labels=[0, 1, 2, 3],
                            target_names=CLASSES, digits=3, zero_division=0))
print("macro F1 (N,S,V) on test:",
      round(f1_score(y_test, pred, labels=[0, 1, 2], average="macro"), 3))

joblib.dump({"model": model, "decision_weights": w, "classes": CLASSES},
            "../models/ecg_rf_small.joblib", compress=3)
print("saved:", round(os.path.getsize("../models/ecg_rf_small.joblib") / 1e6, 2), "MB")

(50969, 57) (49661, 57)
trees=200 leaf= 2 | nodes=  197466 | CV macro F1 (N,S,V)=0.658 | weights=[1. 3. 1. 0.]
trees=100 leaf= 5 | nodes=   72790 | CV macro F1 (N,S,V)=0.658 | weights=[1. 2. 1. 0.]
trees= 50 leaf= 5 | nodes=   36436 | CV macro F1 (N,S,V)=0.663 | weights=[1. 2. 1. 0.]
trees= 50 leaf=20 | nodes=   19760 | CV macro F1 (N,S,V)=0.658 | weights=[1.  1.  0.5 0. ]
trees= 30 leaf=20 | nodes=   11782 | CV macro F1 (N,S,V)=0.660 | weights=[1. 1. 1. 0.]
trees= 30 leaf=50 | nodes=    7378 | CV macro F1 (N,S,V)=0.651 | weights=[1. 1. 1. 0.]

chosen: trees = 30 | leaf = 20 | nodes = 11782 | weights = [1. 1. 1. 0.]
approx. size as compact binary for the phone: 0.26 MB
              precision    recall  f1-score   support

           N      0.961     0.961     0.961     44218
           S      0.187     0.163     0.174      1836
           V      0.808     0.961     0.878      3219
           F      0.000     0.000     0.000       388

    accuracy                          0.924     49

In [2]:
import sys
sys.path.append("..")

import os
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_sample_weight

from src.features import build_features

d = np.load("../data/processed/beats.npz")
CLASSES = ["N", "S", "V", "F"]


def load(split):
    X, y, rr, g = d[f"X_{split}"], d[f"y_{split}"], d[f"rr_{split}"], d[f"g_{split}"]
    m = y != 4                       # drop Q
    return build_features(X[m], rr[m], g[m]), y[m], g[m]


F_train, y_train, g_train = load("train")
F_test, y_test, g_test = load("test")

# 50 trees, leaf 5: best CV score in the table; weights from the same table row
W = np.array([1.0, 2.0, 1.0, 0.0])
model = RandomForestClassifier(n_estimators=50, min_samples_leaf=5,
                               n_jobs=-1, random_state=42)
model.fit(F_train, y_train, sample_weight=compute_sample_weight("balanced", y_train))

nodes = sum(t.tree_.node_count for t in model.estimators_)
print("nodes:", nodes, "(table said 36436)")

proba = np.zeros((len(y_test), 4))
proba[:, model.classes_] = model.predict_proba(F_test)
pred = (proba * W).argmax(1)
print(classification_report(y_test, pred, labels=[0, 1, 2, 3],
                            target_names=CLASSES, digits=3, zero_division=0))
print("macro F1 (N,S,V) on test:",
      round(f1_score(y_test, pred, labels=[0, 1, 2], average="macro"), 3))

joblib.dump({"model": model, "decision_weights": W, "classes": CLASSES},
            "../models/ecg_rf_small.joblib", compress=3)
print("saved:", round(os.path.getsize("../models/ecg_rf_small.joblib") / 1e6, 2), "MB")

nodes: 36436 (table said 36436)
              precision    recall  f1-score   support

           N      0.958     0.972     0.965     44218
           S      0.204     0.162     0.180      1836
           V      0.908     0.944     0.926      3219
           F      0.000     0.000     0.000       388

    accuracy                          0.933     49661
   macro avg      0.518     0.520     0.518     49661
weighted avg      0.920     0.933     0.926     49661

macro F1 (N,S,V) on test: 0.69
saved: 1.27 MB
